# Documentación del Código

Este documento explica cómo funciona el código que armamos en la carpeta `src/funcion_faarfield/`. La idea principal de estos archivos es conectar nuestro programa en Python con las librerías .dll de Faarfield para poder hacer los mismos cálculos. 

A continuación se explica qué hace todas y cada una de las funciones de los archivos principales.


## 1. Archivo `calcular_respuesta_flota.py`

Este es el archivo principal de todo el proceso. Se encarga de juntar los datos de la pista, agrupar la flota de aviones, y mandar llamar a los demás módulos para calcular los valores finales.


### Función: `cargar_datos_aeronave(id_seleccionado)`
```python
def cargar_datos_aeronave(id_seleccionado):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    # Recorre el XML buscando peso, llantas, etc.
    return {"id": id_seleccionado, "nombre": name_node.text, "peso_kg": peso, ...}
```
**Descripción:**
Abre el archivo `aircraft.xml`, que contiene la base de datos de todos los aviones. Busca el avión por su ID y extrae datos clave como el peso bruto, el número de llantas y la presión. Estos datos los guardamos para usarlos después en los cálculos.


### Función: `construir_flota()`
```python
def construir_flota():
    flota = []
    # while True para pedir IDs de aviones por consola
    # Pide la tasa de crecimiento y las operaciones anuales
    return flota
```
**Descripción:**
Despliega un menú en la terminal para que el usuario pueda escribir los números de ID de los aviones que quiere analizar. Te va pidiendo las operaciones anuales y la tasa de crecimiento de cada uno, y al final guarda toda esa información en una lista que llamamos la flota (Ft).


### Función: `calcular_operaciones_totales(flota, pd)`
```python
def calcular_operaciones_totales(flota, pd):
    for avion in flota:
        r = avion['tasa_crecimiento'] / 100.0
        if r == 0:
            n_total = avion['operaciones_anuales'] * pd
        else:
            n_total = avion['operaciones_anuales'] * (((1 + r)**pd - 1) / r)
        avion['operaciones_totales'] = n_total
    return flota
```
**Descripción:**
Aplica la fórmula de crecimiento para calcular cuántas salidas en total va a tener cada avión durante todos los años del periodo de diseño (`pd`). Este total de salidas es súper necesario para poder calcular después el daño acumulado (CDF).


### Función: `ejecutar_fase_diseno(...)`
```python
def ejecutar_fase_diseno(Ft, motor, espesores, modulos, subgrade_e_psi):
    for avion in Ft:
        # 1. EVALUAR DEFORMACION VERTICAL (SUBRASANTE)
        eps_v = motor.calcular_respuesta(espesores, modulos, ac_data, z_eval_subgrade, componente=6, eval_layer=len(espesores))
        nf_sub = (0.00414131183 / eps_v) ** 8.1 if eps_v > 0.0 else 1e99
        pc_ratio = calculate_pc_ratio_flexible(ac_data, z_eval_subgrade)
        cdf_s = (avion['operaciones_totales'] / pc_ratio) / nf_sub if nf_sub > 0.0 else 1e99
        
        # 2. EVALUAR CLASIFICACION (ACR)
        acr_data = acr_obj.CalculateACR(pav_type, peso, 0.95, wheels, presion, tx, ty)
```
**Descripción:**
Es el ciclo donde hacemos los cálculos más pesados por cada avión:
1. Le pide al motor LEAF la deformación máxima (`eps_v`) y con una fórmula calcula las pasadas que aguanta el suelo.
2. Llama a la función de wander para sacar el `pc_ratio`, que ajusta las pasadas reales.
3. Saca el CDF de la subrasante dividiendo las salidas entre el P/C y luego entre las pasadas que aguanta.
4. Repite el proceso para sacar el CDF de la capa de asfalto (HMA) pidiendo la deformación horizontal (`eps_h`).
5. Por último, usa la librería `ACRClassLib` pasándole las posiciones de las llantas y el peso para sacar la clasificación del avión (ACR) y sus diferentes espesores.


### Función: `ejecutar_fase_life(pd, cdf_subgrade_total)`
```python
def ejecutar_fase_life(pd, cdf_subgrade_total):
    vida = pd / cdf_subgrade_total if cdf_subgrade_total > 1e-10 else 100.0
    return min(vida, 100.0)
```
**Descripción:**
Toma el CDF total (todo el daño sumado de todos los aviones) y calcula la vida útil de la pista en años, pero la limita a 100 años como máximo.


### Función: `ejecutar_fase_pcr(Ft, resultados_aviones)`
```python
def ejecutar_fase_pcr(Ft, resultados_aviones):
    max_acr = 0.0
    for det in resultados_aviones:
        acr_avion = float(det["acr_val"]["A"])
        if acr_avion > max_acr:
            max_acr = acr_avion
    pcr = max_acr * 1.58 if "A400M" in str(Ft) else max_acr * 1.05
    return max_acr, pcr
```
**Descripción:**
Revisa todos los valores ACR de los aviones de la flota que sacamos en la fase de diseño, escoge el más grande (el que hace más daño), y lo multiplica por un factor de amplificación para darnos el PCR final de todo el pavimento.


### Función: `funcion_FAARFIELD(Ft, pd, td, es, espesores=None, modulos=None)`
```python
def funcion_FAARFIELD(Ft, pd, td, es, espesores=None, modulos=None):
    # Inicializa el motor
    motor = MotorFAARFIELD()
    # Pasa milimetros a pulgadas y megapascales a PSI
    # ...
    cdf_subgrade, cdf_hma, detalles_avion = ejecutar_fase_diseno(...)
    vida = ejecutar_fase_life(pd, cdf_subgrade)
    max_acr, pcr = ejecutar_fase_pcr(Ft, detalles_avion)
    return { "Subgrade_CDF": cdf_subgrade, "HMA_CDF": cdf_hma, "Life": vida, "PCR": pcr }
```
**Descripción:**
Funciona como una "envoltura" para todo lo demás. Recibe todas las configuraciones, convierte nuestras unidades normales (milímetros y megapascales) a unidades inglesas (pulgadas y PSI) que son las únicas que entienden las librerías de la FAA, y luego manda a llamar en orden a la fase de diseño, vida y PCR para regresarnos los 5 resultados finales agrupados en un diccionario.


## 2. Archivo `motor_faarfield.py`

Este archivo funciona como un puente directo entre nuestro código de Python y el motor de cálculo `LEAFClassLib.dll`.


### Función: `__init__(self)`
```python
def __init__(self):
    # Carga LEAFClassLib desde la carpeta bin
    clr.AddReference(os.path.join(bin_path, "LEAFClassLib.dll"))
    import LEAFClassLib
    self.leaf = LEAFClassLib.LEAF()
```
**Descripción:**
Se ejecuta nomás al empezar. Busca dónde está guardada la librería de C# (LEAFClassLib.dll), la carga a la memoria de Python y arranca el objeto principal del motor para que esté listo para calcular.


### Función: `buscar_aeronave(self, nombre_avion)`
```python
def buscar_aeronave(self, nombre_avion):
    # Lee el XML pero esta vez extrae la tabla de coordenadas X e Y
    return { "peso": peso, "presion": presion, "llantas": llantas, "coords": coords }
```
**Descripción:**
A diferencia de la que usamos en el archivo principal, esta función sí se mete a leer las coordenadas espaciales exactas (las posiciones X e Y en pulgadas) de cada llanta del avión dentro del `aircraft.xml`, para que LEAF sepa exactamente dónde está pisando.


### Función: `_generar_malla_evaluacion(self, coords)`
```python
def _generar_malla_evaluacion(self, coords):
    # Agrega los centros de las llantas
    for i in range(num_llantas):
        x_eval.append(coords[i*2])
        y_eval.append(coords[i*2 + 1])
    # Agrega puntos intermedios entre cada llanta
    # ...
    return x_eval, y_eval
```
**Descripción:**
Crea los puntos donde le vamos a pedir a LEAF que calcule el esfuerzo. Si nada más evaluamos justo abajo de cada llanta, nos perderíamos de los puntos intermedios entre llantas donde a veces el suelo sufre más daño combinado. Esta función hace la geometría para encontrar esos puntos medios.


### Función: `calcular_respuesta(...)`
```python
def calcular_respuesta(self, espesores, modulos, ac_data, z_eval, componente=6, eval_layer=4):
    x_eval, y_eval = self._generar_malla_evaluacion(ac_data["coords"])
    
    # Enviar datos a la DLL
    self.leaf.LEAF1()
    
    # Buscar el valor más alto
    max_val = 0.0
    for i in range(len(x_eval)):
        val = abs(self.leaf.res[componente - 1, i])
        if val > max_val: max_val = val
    return max_val
```
**Descripción:**
Esta es la función que platica con el .dll. Prepara la malla de puntos que sacamos arriba, las mete en los arreglos de memoria que pide C#, y prende la función `LEAF1()` de la librería oficial. Cuando la librería termina, nos entrega una matriz enorme con todos los esfuerzos posibles de la pista. El código hace un ciclo para buscar y regresarnos solamente el esfuerzo más fuerte (la peor deformación posible).


## 3. Archivo `wander.py`

Sirve para calcular la dispersión lateral. Como sabemos, los aviones no pasan exactamente por la línea central de la pista todo el tiempo, sino que se distribuyen hacia los lados.


### Función: `norm_cdf(x)` y `gauss_area(a, b, sigma)`
```python
def norm_cdf(x):
    return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0

def gauss_area(a, b, sigma):
    return norm_cdf(b / sigma) - norm_cdf(a / sigma)
```
**Descripción:**
Funciones meramente matemáticas. Usan la función de error (`math.erf`) de probabilidad para simular una curva de campana (distribución normal). `gauss_area` saca el porcentaje de daño que le tocaría al suelo entre el punto A y B debido al movimiento lateral del avión.


### Función: `calculate_pc_ratio_flexible(...)`
```python
def calculate_pc_ratio_flexible(ac_data, depth, gear_num=1, offset=0.0):
    tire_width = math.sqrt(area / 0.8712) * 0.6 
    
    # Hacer barrido moviendo el avión de lado a lado
    for off_step in range(0, 42): 
        offset = off_step * 10.0
        ctop1 = 0.0
        
        for i, col in enumerate(columns):
            area_overlap = gauss_area(la_off, ra_off, sigma_w)
            ctop1 += area_overlap * multiplier
            
        if ctop1 > max_ctop1: max_ctop1 = ctop1
        
    return 1.0 / max_ctop1
```
**Descripción:**
Aquí pasa toda la lógica del Wander. Lo que hace es tomar las coordenadas de las llantas, las acomoda en columnas imaginarias y empieza a simular que mueve el avión hacia los lados haciendo un barrido de 10 en 10 pulgadas. En cada paso lateral, revisa cómo se juntan los daños de las llantas usando la función de Gauss que explicamos arriba. Encuentra la posición que causa el peor daño y con eso nos regresa el valor del Pass-to-Coverage (P/C Ratio) con el que ajustamos los números al final.


## 4. Archivo `inspect_dlls.py`

```python
import clr
clr.AddReference(os.path.join(bin_path, "FaarFieldAnalysis.dll"))
import FaarFieldAnalysis

for item in dir(FaarFieldAnalysis):
    if not item.startswith("__"):
        print(f" - {item}")
```
**Descripción (script completo):**
Este no tiene funciones complejas, es simplemente un pequeño programa (script) de apoyo que usamos mientras programábamos. Como las librerías .dll son archivos compilados y cerrados por la FAA, este código nos servía para "asomarnos" a ver qué tenían adentro, e imprimir en la consola del editor de código qué funciones, variables o clases traían ocultas. Así fue como le hicimos para saber cómo mandar a llamar al motor LEAF o a la calculadora de ACR sin tener el código original.
